# ddharmon Tutorial: Mapping Biological Entities with BioMapper2

This notebook demonstrates how to use the **ddharmon** package to resolve biological entity names to standardized identifiers using the BioMapper2 API.

## What is BioMapper2?

BioMapper2 is a knowledge graph-backed service that maps human-readable names (like "Glucose" or "BRCA1") to standardized **CURIEs** (Compact URIs). These CURIEs provide unambiguous identifiers that work across databases.

### Understanding CURIEs

A **CURIE** looks like `CHEBI:17234` or `HMDB:HMDB0000122`. It has two parts:
- **Prefix**: The namespace/database (e.g., `CHEBI`, `HMDB`, `UniProtKB`)
- **Local ID**: The identifier within that database

CURIEs solve the "what do you mean by glucose?" problem - instead of ambiguous names, you get precise database references.

### What ddharmon provides

As of **0.4.0**, ddharmon wraps every public endpoint in the BioMapper2 API:

- **Single lookups**: `map_entity("L-Histidine")` → `MappingResult`
- **Batch mapping**: `map_entities([...])` via the native `POST /map/batch` endpoint, auto-chunked at 1000 entities per request
- **Dataset upload**: `map_dataset_file_sync(Path("compounds.tsv"), ...)` streams per-row results back from `POST /map/dataset/stream` — for files too large to materialize as a list
- **Discovery**: `list_entity_types()`, `list_annotators()`, `list_vocabularies()` enumerate what the server supports
- **Typed exceptions + confidence scoring** for production-ready error handling


---

## Part 1: Setup & Health Check

Let's start by importing ddharmon and verifying the API connection.

In [2]:
# Load environment variables from .env file (if present)
from dotenv import load_dotenv
load_dotenv()  # Loads BIOMAPPER_API_KEY from .env file

# Required for running async code in Jupyter notebooks
import nest_asyncio
nest_asyncio.apply()

# Core imports
import ddharmon
from ddharmon import (
    map_entity,
    map_entities,
    summarize,
    BioMapperClient,
    MappingResult,
    MappingSummary,
    BioMapperError,
    BioMapperAuthError,
    BioMapperRateLimitError,
)

print(f"ddharmon version: {ddharmon.__version__}")

ddharmon version: 0.4.0


### API Key Configuration

ddharmon requires a `BIOMAPPER_API_KEY` environment variable. You can set it in several ways:

1. **Environment variable**: `export BIOMAPPER_API_KEY=your_key`
2. **`.env` file**: Create a file with `BIOMAPPER_API_KEY=your_key`
3. **In notebook** (not recommended for shared notebooks):
   ```python
   import os
   os.environ["BIOMAPPER_API_KEY"] = "your_key"
   ```

To request an API key, contact the BioMapper2 team or check your organization's credentials.

### Health Check

Before making mapping requests, let's verify the API is accessible. The `BioMapperClient` provides an async interface with a `health_check()` method.

In [3]:
import asyncio

async def check_health():
    """Verify API connectivity and authentication."""
    async with BioMapperClient() as client:
        return await client.health_check()

health = asyncio.get_event_loop().run_until_complete(check_health())
print(f"API Health: {health}")

API Health: {'status': 'healthy', 'version': '0.1.0', 'mapper_initialized': True}


If you see `True` or a success response, you're ready to go! If you get an authentication error, check your `BIOMAPPER_API_KEY`.

---

## Part 1.5: Discovering what the API supports

Three read-only endpoints let you enumerate the entity types, annotators, and vocabularies the server understands — no need to memorize them from the API docs. Each has a sync free-function form (shown here) and an async method on `BioMapperClient`.


In [4]:
from ddharmon import list_annotators

annotators = list_annotators()
for a in annotators:
    print(f"{a.slug:30s} {a.name}")


kestrel-hybrid-search          KestrelHybridSearchAnnotator
kestrel-text-search            KestrelTextSearchAnnotator
kestrel-vector-search          KestrelVectorSearchAnnotator
metabolomics-workbench         MetabolomicsWorkbenchAnnotator


In [5]:
from ddharmon import list_vocabularies

vocabularies = list_vocabularies()
print(f"{len(vocabularies)} supported vocabularies\n")
for v in vocabularies[:10]:
    print(f"{v.prefix:15s} {v.iri or ''}")


310 supported vocabularies

AEO             http://purl.obolibrary.org/obo/AEO_
AGRKB           https://www.alliancegenome.org/
AHRQ            
APO             http://purl.obolibrary.org/obo/APO_
ATC             https://www.whocc.no/atc_ddd_index/?code=
AraPort         https://www.arabidopsis.org/servlets/TairObject?accession=
AspGD           http://www.aspergillusgenome.org/cgi-bin/locus.pl?dbid=
BFO             http://purl.obolibrary.org/obo/BFO_
BIGG.METABOLITE http://identifiers.org/bigg.metabolite/
BIGG.REACTION   http://identifiers.org/bigg.reaction/


In [6]:
from ddharmon import list_entity_types

entity_types = list_entity_types()
for et in entity_types:
    aliases = ', '.join(et.aliases) if et.aliases else '(no aliases)'
    print(f"{et.type}\n    aliases: {aliases}\n")


biolink:ClinicalFinding
    aliases: clinicallab, lab

biolink:Disease
    aliases: disease

biolink:Drug
    aliases: drug

biolink:Gene
    aliases: gene

biolink:Pathway
    aliases: pathway

biolink:PhenotypicFeature
    aliases: phenotype

biolink:Protein
    aliases: protein

biolink:SmallMolecule
    aliases: lipid, metabolite



---

## Part 2: Single Entity Lookup

The simplest operation is mapping a single entity name. Let's look up "L-Histidine", an amino acid.

### Using `map_entity()`

The `map_entity()` function takes a name string and returns a `MappingResult` object with all the mapping details.

In [7]:
result = map_entity("L-Histidine")

print("=== MappingResult Fields ===")
print(f"Query Name:       {result.query_name}")
print(f"Resolved:         {result.resolved}")
print(f"Primary CURIE:    {result.primary_curie}")
print(f"Chosen KG ID:     {result.chosen_kg_id}")
print(f"Confidence Score: {result.confidence_score}")
print(f"Confidence Tier:  {result.confidence_tier}")
print()
print("=== Vocabulary-Specific IDs ===")
print(f"CHEBI IDs:        {result.ids_for('CHEBI')}")
print(f"HMDB IDs:         {result.ids_for('HMDB')}")
print(f"RefMet IDs:       {result.ids_for('refmet_id')}")

=== MappingResult Fields ===
Query Name:       L-Histidine
Resolved:         True
Primary CURIE:    RM:0129894
Chosen KG ID:     CHEBI:15971
Confidence Score: 2.4898567923359374
Confidence Tier:  high

=== Vocabulary-Specific IDs ===
CHEBI IDs:        ['15971']
HMDB IDs:         []
RefMet IDs:       ['RM0129894']


### Understanding the MappingResult

| Field | Description |
|-------|-------------|
| `query_name` | The original name you searched for |
| `resolved` | Boolean - did BioMapper2 find a match? |
| `primary_curie` | The best/canonical CURIE identifier |
| `chosen_kg_id` | Internal knowledge graph node ID |
| `confidence_score` | Numeric confidence (higher = better) |
| `confidence_tier` | Human-readable tier: high/medium/low/unknown |
| `identifiers` | Dictionary of all mapped vocabularies |

### Confidence Tiers

The `confidence_tier` property categorizes the numeric score:

- **high** (≥2.0): Strong match, use with confidence
- **medium** (1.0-2.0): Good match, may want to verify
- **low** (<1.0): Weak match, manual review recommended
- **unknown**: No confidence score available

### Async Client Usage

For more control (or when integrating into async code), use the `BioMapperClient` directly:

In [8]:
async def map_with_client():
    """Demonstrate async client usage."""
    async with BioMapperClient() as client:
        result = await client.map_entity("Glucose")
        return result

async_result = asyncio.get_event_loop().run_until_complete(map_with_client())
print(f"Glucose -> {async_result.primary_curie} (confidence: {async_result.confidence_tier})")

Glucose -> MESH:D001786 (confidence: medium)


## Part 3: Batch Metabolite Mapping

Real-world metabolomics data typically involves hundreds or thousands of compounds. The `map_entities()` function handles batches efficiently by going through the native `POST /map/batch` endpoint — 100 records cost **one** round-trip instead of 100. Inputs are auto-chunked at 1000 entities per request.


### Toy Metabolite Dataset

Let's create a small dataset that represents typical metabolomics challenges:
- Well-known compound names
- Names with HMDB identifier hints
- Obscure or ambiguous names
- **Vendor codes and domain notation** that will match with low confidence (false positives)

In [9]:
metabolite_records = [
    # Clean, well-known names (should resolve with high confidence)
    {"name": "Glucose"},
    {"name": "L-Carnitine"},
    {"name": "Sphinganine"},
    {"name": "Choline"},
    {"name": "Creatinine"},
    {"name": "Taurine"},
    {"name": "Uric acid"},

    # With HMDB hint (improves resolution)
    {"name": "Histidine", "identifiers": {"HMDB": "HMDB0000177"}},

    # Obscure names (medium/low confidence expected)
    {"name": "2-Hydroxyglutaric acid"},
    {"name": "3-Methylhistidine"},
    {"name": "Dimethylglycine"},

    # Vendor codes & domain notation (will match with LOW confidence - false positives)
    {"name": "SCHEMBL20255602"},   # Patent-derived ID (SureChEMBL)
    {"name": "EN300-1697854"},     # Enamine catalog code
    {"name": "STL218830"},         # STL catalog code
    {"name": "Val-C20:0"},         # Lipid shorthand notation
]

print(f"Dataset: {len(metabolite_records)} compounds")

Dataset: 15 compounds


### Running the Batch

The `map_entities()` function processes records sequentially with a configurable delay between requests (default: 0.3s). Set `progress=True` to see a progress bar.

> **New in 0.3.0:** batch mapping now goes through the native `POST /map/batch` endpoint, so 100 records cost **one** round-trip instead of 100. Inputs are auto-chunked at 1000 entities per request. The `rate_limit_delay` parameter still exists but now applies *between chunks* rather than between individual records; it will be removed in 1.0.0.

In [10]:
# Map all metabolites with progress bar
results = map_entities(metabolite_records, progress=True)

print(f"\nMapped {len(results)} compounds")


Mapped 15 compounds


### Summarizing Results

The `summarize()` function provides aggregate statistics about your batch mapping.

In [11]:
summary = summarize(results)

print("=== Mapping Summary ===")
print(f"Total queries:     {summary.total_queried}")
print(f"Resolved:          {summary.resolved} ({summary.resolution_rate:.1%})")
print(f"Unresolved:        {summary.unresolved}")
print(f"Errors:            {summary.errors}")
print()
print("=== Vocabulary Coverage ===")
for vocab, count in sorted(summary.vocabulary_coverage.items()):
    print(f"  {vocab}: {count}")

=== Mapping Summary ===
Total queries:     15
Resolved:          15 (100.0%)
Unresolved:        0
Errors:            0

=== Vocabulary Coverage ===
  CHEBI: 4
  FOODON: 1
  MESH: 1
  PUBCHEM.COMPOUND: 1
  UMLS: 7
  refmet_id: 10


### Results Table

Let's view all results in a tabular format:

In [12]:
print(f"{'Query Name':<25} {'Primary CURIE':<25} {'Tier':<10}")
print("=" * 60)

for r in results:
    curie = r.primary_curie or "(not resolved)"
    tier = r.confidence_tier if r.resolved else "—"
    print(f"{r.query_name:<25} {curie:<25} {tier:<10}")

Query Name                Primary CURIE             Tier      
Glucose                   MESH:D001786              medium    
L-Carnitine               CHEBI:16347               low       
Sphinganine               UMLS:C0074987             high      
Choline                   RM:0028739                high      
Creatinine                CHEBI:16737               high      
Taurine                   RM:0118223                high      
Uric acid                 RM:0160856                high      
Histidine                 HMDB:HMDB0000177          unknown   
2-Hydroxyglutaric acid    UMLS:C0093153             high      
3-Methylhistidine         UMLS:C0047600             high      
Dimethylglycine           RM:0135883                high      
SCHEMBL20255602           UMLS:C5544974             low       
EN300-1697854             UMLS:C1709040             low       
STL218830                 UMLS:C3661006             low       
Val-C20:0                 PUBCHEM.COMPOUND:10431989 low

---

## Part 4: Gene/Protein Mapping

ddharmon can also map gene and protein names. The key difference is specifying `entity_type="biolink:Gene"` instead of the default `"biolink:SmallMolecule"`.

### Toy Gene Dataset

In [13]:
gene_records = [
    {"name": "BRCA1"},   # Breast cancer susceptibility gene
    {"name": "TP53"},    # Tumor suppressor
    {"name": "EGFR"},    # Epidermal growth factor receptor
    {"name": "TNF"},     # Tumor necrosis factor
    {"name": "IL6"},     # Interleukin 6
    {"name": "APOE"},    # Apolipoprotein E
    {"name": "ACE2"},    # Angiotensin-converting enzyme 2
    {"name": "PCSK9"},   # Proprotein convertase
]

print(f"Dataset: {len(gene_records)} genes")

Dataset: 8 genes


### Mapping with entity_type

The `entity_type` parameter tells BioMapper2 how to route the query. For genes/proteins, use `"biolink:Gene"` or `"biolink:Protein"`.

In [14]:
# Map genes (note the entity_type parameter)
gene_results = map_entities(
    gene_records,
    entity_type="biolink:Gene",
    progress=True
)

# Show summary
gene_summary = summarize(gene_results)
print(f"\nResolved: {gene_summary.resolved}/{gene_summary.total_queried}")
print(f"\nVocabulary coverage: {gene_summary.vocabulary_coverage}")


Resolved: 8/8

Vocabulary coverage: {'NCBIGene': 7, 'ENSEMBL': 1}


### Gene Mapping Results

Gene mappings return different vocabularies than metabolites:
- **Metabolites**: CHEBI, HMDB, RefMet, PubChem, etc.
- **Genes**: NCBI Gene, Ensembl, HGNC, etc.

**Note:** The BioMapper2 API currently returns gene identifiers (NCBIGene, ENSEMBL) but does not include UniProt protein mappings. If you need UniProt IDs, you'll need to use a separate ID conversion service like UniProt's ID Mapping tool.

In [15]:
print(f"{'Gene':<10} {'Primary CURIE':<30}")
print("=" * 40)

for r in gene_results:
    curie = r.primary_curie or "(not resolved)"
    print(f"{r.query_name:<10} {curie:<30}")

Gene       Primary CURIE                 
BRCA1      NCBIGene:100049662            
TP53       NCBIGene:101833915            
EGFR       NCBIGene:397070               
TNF        ENSEMBL:ENSG00000228978       
IL6        NCBIGene:280826               
APOE       NCBIGene:397576               
ACE2       NCBIGene:100327258            
PCSK9      NCBIGene:100620501            


---

## Part 5: Error Handling & Edge Cases

Production code should handle potential errors gracefully. ddharmon provides specific exception classes for different failure modes.

### Exception Classes

| Exception | When it's raised |
|-----------|------------------|
| `BioMapperError` | Base class for all ddharmon errors |
| `BioMapperAuthError` | Invalid or missing API key |
| `BioMapperRateLimitError` | Too many requests (429 response) |
| `BioMapperTimeoutError` | Request timed out |
| `BioMapperServerError` | Server-side error (5xx response) |

### Recommended Error Handling Pattern

In [16]:
from ddharmon import (
    BioMapperError,
    BioMapperAuthError,
    BioMapperRateLimitError,
    BioMapperTimeoutError,
)

def safe_map_entity(name: str) -> MappingResult | None:
    """Map an entity with proper error handling."""
    try:
        return map_entity(name)
    except BioMapperAuthError:
        print(f"Authentication failed - check BIOMAPPER_API_KEY")
        return None
    except BioMapperRateLimitError as e:
        print(f"Rate limited! Retry after {e.retry_after}s")
        return None
    except BioMapperTimeoutError:
        print(f"Request timed out for '{name}'")
        return None
    except BioMapperError as e:
        print(f"Mapping error for '{name}': {e}")
        return None

# Test with a valid query
result = safe_map_entity("Caffeine")
if result:
    print(f"Success: {result.query_name} -> {result.primary_curie}")

Success: Caffeine -> RM:0032992


### Low-Confidence Matches (Potential False Positives)

BioMapper2 uses fuzzy matching to maximize coverage, which means vendor codes and malformed names often get *some* match - but with low confidence scores. These are effectively **false positives** that need manual review.

Let's examine the vendor codes from our batch:

In [17]:
# Find low-confidence matches that are likely false positives
low_confidence = [r for r in results if r.confidence_tier == "low"]

if low_confidence:
    print("=== Low-Confidence Matches (Review Required) ===\n")
    for r in low_confidence:
        print(f"Query Name:       {r.query_name}")
        print(f"Matched To:       {r.primary_curie}")
        print(f"Confidence Score: {r.confidence_score:.2f}" if r.confidence_score else "Confidence Score: N/A")
        print(f"Confidence Tier:  {r.confidence_tier}")
        print()
    print(f"⚠️  {len(low_confidence)} compounds matched with LOW confidence.")
    print("   These are likely spurious matches to unrelated entities.")
    print("   In production, filter these out or flag for manual review.")
else:
    print("All entities matched with medium or high confidence!")

=== Low-Confidence Matches (Review Required) ===

Query Name:       L-Carnitine
Matched To:       CHEBI:16347
Confidence Score: 0.90
Confidence Tier:  low

Query Name:       SCHEMBL20255602
Matched To:       UMLS:C5544974
Confidence Score: 0.89
Confidence Tier:  low

Query Name:       EN300-1697854
Matched To:       UMLS:C1709040
Confidence Score: 0.73
Confidence Tier:  low

Query Name:       STL218830
Matched To:       UMLS:C3661006
Confidence Score: 0.81
Confidence Tier:  low

Query Name:       Val-C20:0
Matched To:       PUBCHEM.COMPOUND:10431989
Confidence Score: 0.67
Confidence Tier:  low

⚠️  5 compounds matched with LOW confidence.
   These are likely spurious matches to unrelated entities.
   In production, filter these out or flag for manual review.


### Controlling Annotator Behavior (Advanced)

BioMapper2 uses multiple **annotators** (resolvers) to map names to identifiers. You can restrict which annotators are used with the `annotators` parameter:

| Annotator | Description |
|-----------|-------------|
| `"kestrel-vector-search"` | Embedding-based semantic search |
| `"kestrel-hybrid-search"` | Combines vector + keyword search |
| `"metabolomics-workbench"` | RefMet-based resolution only |

This is useful when you want results from a specific vocabulary (e.g., RefMet only).

In [18]:
# Compare default (all annotators) vs RefMet-only behavior
# Using metabolomics-workbench on vendor codes demonstrates truly unresolved results
vendor_codes = [
    {"name": "SCHEMBL20255602"},  # SureChEMBL patent-derived ID
    {"name": "EN300-1697854"},    # Enamine catalog code
]

print("=== Default Behavior (all annotators) ===")
default_results = map_entities(vendor_codes, progress=False)
for r in default_results:
    status = f"resolved -> {r.primary_curie}" if r.resolved else "UNRESOLVED"
    print(f"  {r.query_name}: {status} (conf: {r.confidence_tier})")

print("\n=== RefMet-Only (metabolomics-workbench) ===")
print("Vendor codes aren't in RefMet, so they return truly unresolved:")
refmet_results = map_entities(
    vendor_codes,
    annotators=["metabolomics-workbench"],  # RefMet vocabulary only
    progress=False
)
for r in refmet_results:
    status = f"resolved -> {r.primary_curie}" if r.resolved else "UNRESOLVED"
    print(f"  {r.query_name}: {status} (conf: {r.confidence_tier})")

=== Default Behavior (all annotators) ===
  SCHEMBL20255602: resolved -> UMLS:C5544974 (conf: low)
  EN300-1697854: resolved -> UMLS:C1709040 (conf: low)

=== RefMet-Only (metabolomics-workbench) ===
Vendor codes aren't in RefMet, so they return truly unresolved:
  SCHEMBL20255602: UNRESOLVED (conf: unknown)
  EN300-1697854: UNRESOLVED (conf: unknown)


**When to use `annotators`:**

| Scenario | Recommended Setting |
|----------|---------------------|
| General metabolomics | Default (all annotators) + filter by confidence |
| RefMet-focused workflow | `annotators=["metabolomics-workbench"]` |
| Need truly unresolved for unknown compounds | `annotators=["metabolomics-workbench"]` |
| Maximum coverage | Default (all annotators) |

**Note:** Using `metabolomics-workbench` on vendor codes produces **truly unresolved** results
since those codes aren't in the RefMet vocabulary. This is useful when you need a binary
resolved/unresolved classification rather than low-confidence fuzzy matches to UMLS.

### Common Gotchas

**1. Rate Limiting**

If you hit a 429, catch `BioMapperRateLimitError` and honor the server's `retry_after` hint rather than throttling client-side. The `rate_limit_delay` parameter on `map_entities` still exists but is deprecated and scheduled for removal in 1.0.0 — the native batch endpoint already moves work off the per-request path:

```python
try:
    results = map_entities(records, progress=True)
except BioMapperRateLimitError as e:
    time.sleep(e.retry_after or 1.0)
    results = map_entities(records, progress=True)
```

**2. Vendor Codes Match Spuriously**

Internal codes like `SCHEMBL20255602` or `Val-C20:0` aren't in public knowledge graphs, but BioMapper2's fuzzy matching may still return *some* match with **low confidence**. Always filter by confidence tier:
```python
# Only trust high/medium confidence matches
reliable = [r for r in results if r.confidence_tier in ("high", "medium")]
needs_review = [r for r in results if r.confidence_tier == "low"]
```

**3. Entity Type Matters**

Fuzzy matching may return *something* for "ACE2" under `SmallMolecule`, but it won't be the gene. Always use the appropriate `entity_type` for reliable matching:
- Metabolites: `"biolink:SmallMolecule"` (default)
- Genes: `"biolink:Gene"`
- Proteins: `"biolink:Protein"`

**4. HMDB Hints Improve Resolution**

If you have partial identifiers, include them:
```python
{"name": "Histidine", "identifiers": {"HMDB": "HMDB0000177"}}
```


---

## Part 6: Working with Results

Once you have mapping results, you'll often need to filter, extract, and analyze them.

### Filtering by Confidence

Separate high-confidence results from those needing review.

**Note:** With default annotators, `unresolved` will typically be 0 because BioMapper2's fuzzy UMLS matching provides *some* match for almost any input. The key is that vendor codes and gibberish get **low-confidence** matches. Use `confidence_tier` filtering (not `resolved`) to identify problematic results. If you need truly unresolved results, use `annotators=["metabolomics-workbench"]` as shown above.

In [19]:
# Filter by confidence tier
high_confidence = [r for r in results if r.confidence_tier == "high"]
needs_review = [r for r in results if r.confidence_tier in ("medium", "low")]
unresolved = [r for r in results if not r.resolved]

print(f"High confidence:  {len(high_confidence)} compounds")
print(f"Needs review:     {len(needs_review)} compounds")
print(f"Unresolved:       {len(unresolved)} compounds")

if needs_review:
    print(f"\nCompounds to review: {[r.query_name for r in needs_review]}")

High confidence:  8 compounds
Needs review:     6 compounds
Unresolved:       0 compounds

Compounds to review: ['Glucose', 'L-Carnitine', 'SCHEMBL20255602', 'EN300-1697854', 'STL218830', 'Val-C20:0']


### Extracting Vocabulary-Specific IDs

Often you need IDs for a specific database (e.g., CHEBI for pathway tools, HMDB for metabolomics databases):

In [20]:
# Build a mapping: compound name -> CHEBI IDs
chebi_mapping = {
    r.query_name: r.ids_for("CHEBI") 
    for r in results 
    if r.resolved and r.ids_for("CHEBI")
}

print("CHEBI ID Mapping:")
for name, chebi_ids in list(chebi_mapping.items())[:5]:
    print(f"  {name}: {chebi_ids}")

CHEBI ID Mapping:
  L-Carnitine: ['16347']
  Choline: ['3668']
  Creatinine: ['16737']
  Uric acid: ['17775']


### Vocabulary Coverage Analysis

The summary's `vocabulary_coverage` shows which databases had identifiers:

In [21]:
# Get summary for resolved results only
resolved_results = [r for r in results if r.resolved]
resolved_summary = summarize(resolved_results)

print(f"Vocabulary coverage across {resolved_summary.total_queried} resolved compounds:")
print()

# Sort by count
for vocab, count in sorted(
    resolved_summary.vocabulary_coverage.items(),
    key=lambda x: x[1],
    reverse=True
):
    pct = count / resolved_summary.total_queried * 100
    bar = "█" * int(pct // 5)
    print(f"  {vocab:<15} {count:>3} ({pct:5.1f}%) {bar}")

Vocabulary coverage across 15 resolved compounds:

  refmet_id        10 ( 66.7%) █████████████
  UMLS              7 ( 46.7%) █████████
  CHEBI             4 ( 26.7%) █████
  MESH              1 (  6.7%) █
  FOODON            1 (  6.7%) █
  PUBCHEM.COMPOUND   1 (  6.7%) █


---

## Part 7: Dataset Upload (synchronous + async)

For larger inputs — hundreds to tens of thousands of rows — hand the server a TSV/CSV file directly instead of materializing a list. The server processes the file row-by-row over `POST /map/dataset/stream` and emits NDJSON results back; the client streams them.

Two flavors:

- **Sync (notebook-friendly):** `map_dataset_file_sync(...)` blocks and returns a completed `DatasetMappingResult`. With `progress=True` you get a tqdm bar; with `total_hint` it shows percentage.
- **Async (live-stream-into-UI friendly):** `BioMapperClient.map_dataset_file_iter(...)` is an async generator that yields each `MappingResult` as it arrives.


### Sync upload with progress + raise_for_error


In [22]:
from pathlib import Path

from ddharmon import map_dataset_file_sync, summarize

fixture = Path("../tests/fixtures/metabolites_sample.tsv")

result = map_dataset_file_sync(
    fixture,
    name_column="name",
    provided_id_columns=["hmdb_id"],
    progress=True,
    total_hint=10,
)

# Opt-in: turn a captured mid-stream failure back into an exception.
# Skipping this is a silent-data-loss footgun on truncated streams.
result.raise_for_error()

summary = summarize(result.results)
print(f"resolved {summary.resolved}/{summary.total_queried}")
for r in result.results[:5]:
    print(f"  {r.query_name:<20} -> {r.primary_curie}")


resolved 10/10
  L-Histidine          -> HMDB:HMDB0000177
  Glucose              -> HMDB:HMDB0000122
  Pyruvic acid         -> HMDB:HMDB0000243
  L-Alanine            -> HMDB:HMDB0000161
  Citric acid          -> HMDB:HMDB0000094


### Async iterator — per-result streaming


The async iterator is the right primitive when you want to render results as they arrive (live dashboards, web UIs, anything interactive). The notebook uses `nest_asyncio` so we can `await` directly in a cell.


In [23]:
import nest_asyncio
nest_asyncio.apply()

from ddharmon import BioMapperClient

async def stream_first_five():
    async with BioMapperClient() as c:
        seen = 0
        async for r in c.map_dataset_file_iter(
            fixture,
            name_column="name",
            provided_id_columns=["hmdb_id"],
        ):
            print(f"  arrived: {r.query_name:<20} -> {r.primary_curie}")
            seen += 1
            if seen >= 5:
                break

await stream_first_five()


  arrived: L-Histidine          -> HMDB:HMDB0000177
  arrived: Glucose              -> HMDB:HMDB0000122
  arrived: Pyruvic acid         -> HMDB:HMDB0000243
  arrived: L-Alanine            -> HMDB:HMDB0000161
  arrived: Citric acid          -> HMDB:HMDB0000094


### The `DatasetMappingResult` return type

`map_dataset_file_sync` returns a `DatasetMappingResult` with these fields:

| Attribute | Type | Description |
|---|---|---|
| `results` | `list[MappingResult]` | Per-row outcomes, in server-emitted order |
| `stats` | `dict[str, Any]` | Server summary. Empty `{}` unless the stream emits a terminal summary line |
| `metadata` | `ApiMetadata` | Request metadata; stays at defaults when the stream truncates before completion |
| `error` | `str \| None` | Mid-stream transport failure text. `None` on clean runs |

**Pick your shape.** The type mirrors `httpx.Response` — choose exception semantics or partial-results semantics depending on what your caller wants:

```python
# Exception semantics (notebooks, scripts)
result = map_dataset_file_sync(path, name_column="name", provided_id_columns=["hmdb_id"])
result.raise_for_error()           # raises BioMapperError if .error is set
summary = summarize(result.results)

# Partial-results semantics (UIs, resumable workflows)
result = map_dataset_file_sync(path, name_column="name", provided_id_columns=["hmdb_id"])
if result.error:
    log_and_render_partial(result.results, result.error)
else:
    render_complete(result.results)
```

Silent consumption — using `.results` without calling `raise_for_error()` and without inspecting `.error` — is the main footgun this model is designed to prevent.

> **Gotcha: `confidence_score` is always `None` for dataset-stream results.**
> The `/map/dataset/stream` endpoint emits a slimmer per-row payload than `/map/batch` — no `assigned_ids` block, so no annotator scores. Use `map_entity` / `map_entities` if you need confidence tiers. All other `MappingResult` fields (`resolved`, `primary_curie`, `chosen_kg_id`, `identifiers`, `ids_for()`) work normally.


---

## Part 8: Next Steps

You've now seen every endpoint ddharmon wraps in 0.4.0 — single lookup, native batch, file-based dataset upload, and the three discovery endpoints. That covers **full Python-API parity with biomapper2** as of this release.

### On the roadmap

- **0.5.0 — Typer-based CLI.** Adds `ddharmon map`, `ddharmon batch`, `ddharmon annotators`, etc. as convenience sugar over the Python API. Installed as an optional extra: `pip install "ddharmon[cli]"`.
- **1.0.0 — Stability cut.** Removes the deprecated `rate_limit_delay` parameter on `map_entities`, publishes the symbol-stability contract in the README, and declares the Python API frozen.

### Where to go from here

- **Custom pipelines:** the async iterator (`BioMapperClient.map_dataset_file_iter`) is the primitive the Entity Linker UI uses to stream results into a live dashboard. Any async framework can plug in the same way.
- **Preprocessing:** `ddharmon.extras.metabolon` ships `clean_compound_name` and `extract_hmdb_id` for vendor-format inputs.
- **Issues, API requests, new endpoint wrappers:** [github.com/trentleslie/ddharmon/issues](https://github.com/trentleslie/ddharmon/issues)
